In [2]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import pygame
import os
import random

# Paths
model_path = r"C:\Users\Bushra Shahid\Desktop\IDVS\FER Model\model\fer_best_model.keras"
songs_path = r"C:\Users\Bushra Shahid\Desktop\IDVS\FER Model\songs"

# Class labels
CLASSES = ['happiness', 'neutral', 'sadness']

# Load trained FER model
fer_model = load_model(model_path)

# Initialize pygame for music
pygame.mixer.init()

print("Model loaded successfully!")
print("Pygame initialized!")

Model loaded successfully!
Pygame initialized!


In [2]:
# Load OpenCV face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Track current playing emotion
current_emotion = None
current_song = None

def get_random_song(emotion):
    """Get a random song from emotion folder"""
    emotion_folder = os.path.join(songs_path, emotion)
    if not os.path.exists(emotion_folder):
        return None
    songs = [f for f in os.listdir(emotion_folder) if f.endswith('.mp3')]
    if not songs:
        return None
    return os.path.join(emotion_folder, random.choice(songs))

def play_song(emotion):
    """Play song based on detected emotion"""
    global current_emotion, current_song
    if emotion != current_emotion:
        song_path = get_random_song(emotion)
        if song_path:
            pygame.mixer.music.load(song_path)
            pygame.mixer.music.play(-1)
            current_song = song_path
            print(f"Playing song for: {emotion}")
        else:
            print(f"No songs found for: {emotion} — add songs later!")
        current_emotion = emotion

def predict_emotion(face_img):
    """Predict emotion with adjusted thresholds"""
    face_resized = cv2.resize(face_img, (48, 48))
    face_normalized = face_resized / 255.0
    face_input = face_normalized.reshape(1, 48, 48, 1)

    prediction = fer_model.predict(face_input, verbose=0)
    happiness_conf = prediction[0][0]
    neutral_conf   = prediction[0][1]
    sadness_conf   = prediction[0][2]

    # Happiness clearly winning
    if happiness_conf > 0.60:
        return 'happiness', happiness_conf

    # Sadness boost - lowered threshold to 0.15
    if sadness_conf > 0.15:
        return 'sadness', sadness_conf

    return 'neutral', neutral_conf

# Emotion history for smoothing
emotion_history = []
HISTORY_SIZE = 5

def get_stable_emotion(emotion):
    """Smooth prediction using last 5 frames"""
    emotion_history.append(emotion)
    if len(emotion_history) > HISTORY_SIZE:
        emotion_history.pop(0)
    return max(set(emotion_history), key=emotion_history.count)

print("Functions ready!")

Functions ready!


In [3]:
# Start camera
cap = cv2.VideoCapture(0)

print("Camera started! Press Q to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(48, 48)
    )

    for (x, y, w, h) in faces:
        # Crop face
        face_img = gray[y:y+h, x:x+w]

        # Predict emotion
        emotion, confidence = predict_emotion(face_img)

        # Stabilize emotion
        stable_emotion = get_stable_emotion(emotion)

        # Play song
        play_song(stable_emotion)

        # Box color per emotion
        color = (0, 255, 0)   if stable_emotion == 'happiness' else \
                (0, 165, 255) if stable_emotion == 'neutral'   else \
                (0, 0, 255)

        # Draw face box
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

        # Show emotion + confidence
        label = f"{stable_emotion} ({confidence:.0%})"
        cv2.putText(frame, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # Show frame
    cv2.imshow('FER - Emotion Detection', frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()
pygame.mixer.music.stop()
print("Camera stopped!")

Camera started! Press Q to quit.
Playing song for: neutral
Playing song for: happiness
Playing song for: neutral
Playing song for: happiness
Playing song for: neutral
Camera stopped!
